# Notebook 3 – Missing Value Handling



## 1. What are missing values?
Missing values are observations for which a value is unavailable, unknown, not recorded, or not applicable. In pandas, missing data may appear as `NaN`, `None`, `NaT`, or `pd.NA`. Empty strings and special text such as `"unknown"` are not automatically treated as missing until standardized.

## 2. Why missing values occur
- Data-entry or sensor failures
- Survey non-response
- Data not collected by a source system
- A field being not applicable
- Merging records from different sources
- Privacy restrictions
- Values intentionally hidden because of the outcome or measurement process

The reason matters: missingness may itself contain useful information or introduce bias.

## 3. MCAR, MAR, and MNAR
- **MCAR (Missing Completely At Random):** Missingness is unrelated to observed or unobserved data. Example: a random system outage loses a few records.
- **MAR (Missing At Random):** Missingness can be explained by observed variables. Example: income is more often missing for a particular age group, and age is observed.
- **MNAR (Missing Not At Random):** Missingness depends on the missing value itself or unobserved factors. Example: people with very high incomes are less likely to disclose income.

These mechanisms cannot always be distinguished from the observed data alone. Use domain knowledge and sensitivity analysis.

## 4. Detect missing values and percentages

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [22, 35, np.nan, 48, 29, np.nan],
    "income": [25000, np.nan, 33000, 71000, np.nan, 42000],
    "city": ["Chennai", "Salem", None, "Madurai", "Chennai", None],
    "score": [70, 80, 75, np.nan, 90, 85]
})

display(df)
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_pct
}).sort_values("missing_percent", ascending=False)
display(missing_report)
print("Total missing cells:", int(df.isna().sum().sum()))

## 5. Dropping rows
**Use when:** only a small number of rows are missing, the rows are not essential, and deletion is unlikely to bias the sample.

**Avoid when:** missingness is substantial or systematic, the dataset is small, or deleting rows disproportionately removes a group.

**Advantages:** simple; no imputed values are introduced.

**Limitations:** reduces sample size and may bias results. Always compare row counts and the composition of retained vs removed data.

In [ ]:
complete_rows = df.dropna(subset=["age", "income"])
print("Original rows:", len(df), "| Remaining rows:", len(complete_rows))
display(complete_rows)

## 6. Dropping columns
**Use when:** a feature is mostly missing, has little practical value, or cannot be recovered reliably.

**Avoid when:** the feature is important, missingness is informative, or enough values remain for a defensible method.

**Advantages:** easy and may simplify analysis.

**Limitations:** loses potentially useful signal and can create bias. There is no universal missing-percentage threshold; decide using context and validation.

In [ ]:
column_missing_pct = df.isna().mean() * 100
print(column_missing_pct.sort_values(ascending=False))
# Example only: review before dropping; do not use a threshold blindly.

## 7. Mean imputation
**Use when:** a numerical feature is roughly symmetric, extreme values are limited, and a simple baseline is appropriate.

**Avoid when:** the distribution is skewed, outliers are strong, or preserving variance/relationships is important.

**Advantages:** simple and fast.

**Limitations:** pulls values toward the mean, reduces variance, and can weaken correlations. Fit the mean on training data only.

In [ ]:
from sklearn.impute import SimpleImputer

mean_imputer = SimpleImputer(strategy="mean")
# Fit on training data in real workflows; this tiny example is for demonstration.
numeric_mean = mean_imputer.fit_transform(df[["score"]])
print(pd.Series(numeric_mean.ravel(), name="score_mean_imputed"))

## 8. Median imputation
**Use when:** numerical data are skewed or contain outliers; the median is more robust than the mean.

**Avoid when:** the missingness mechanism is informative or relationships between multiple features need to be preserved.

**Advantages:** robust, simple, and resistant to extreme values.

**Limitations:** reduces variability and may distort relationships; it does not account for uncertainty in imputed values.

In [ ]:
median_imputer = SimpleImputer(strategy="median")
display(pd.DataFrame(
    median_imputer.fit_transform(df[["income"]]),
    columns=["income_median_imputed"]
))

## 9. Mode imputation
**Use when:** a categorical feature has a clear common category and a simple baseline is sufficient.

**Avoid when:** categories are highly imbalanced, missingness is informative, or filling the dominant category would create a misleading majority.

**Advantages:** easy and preserves valid category labels.

**Limitations:** increases the apparent frequency of the most common category and can understate uncertainty.

In [ ]:
mode_imputer = SimpleImputer(strategy="most_frequent")
city_filled = mode_imputer.fit_transform(df[["city"]])
display(pd.DataFrame(city_filled, columns=["city_mode_imputed"]))

## 10. Constant-value imputation
**Use when:** a value such as `"Unknown"` is a meaningful category, or a documented domain-specific sentinel is appropriate.

**Avoid when:** the constant could be mistaken for a real measurement, such as replacing unknown income with `0` without justification.

**Advantages:** transparent and can preserve the fact that a value was missing.

**Limitations:** creates an artificial group/value and can affect model behavior. Consider adding a missingness indicator.

In [ ]:
constant_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
city_constant = constant_imputer.fit_transform(df[["city"]])
display(pd.DataFrame(city_constant, columns=["city_constant_imputed"]))

## 11. Forward fill and backward fill
- **Forward fill (`ffill`):** use the last observed value before a gap.
- **Backward fill (`bfill`):** use the next observed value after a gap.

**Use when:** records are ordered and values reasonably persist across adjacent observations, such as some sensor or operational time series.

**Avoid when:** row order is not meaningful, observations are unrelated, or using a future value would leak information into a real-time prediction.

**Advantages:** simple and preserves observed values.

**Limitations:** can propagate stale values, create long artificial runs, and introduce look-ahead leakage with backward fill. Sort by entity and time first; consider a limit on consecutive fills.

In [ ]:
series = pd.Series([10.0, np.nan, np.nan, 16.0, np.nan])
print("Original:", series.tolist())
print("Forward fill:", series.ffill().tolist())
print("Backward fill:", series.bfill().tolist())

## 12. Interpolation
Interpolation estimates missing values between observed points, often using a linear trend.

**Use when:** observations have a meaningful numeric/time order and values are expected to change smoothly between nearby observations.

**Avoid when:** data are categorical, gaps are long, abrupt changes are common, or the method would use future values unavailable at prediction time.

**Advantages:** can preserve a smooth local trend and be useful for sensor/time-series data.

**Limitations:** estimates may be unrealistic around spikes, boundaries, or regime changes. Validate against domain knowledge.

In [ ]:
time_series = pd.Series(
    [10.0, np.nan, 14.0, np.nan, 22.0],
    index=pd.date_range("2026-01-01", periods=5, freq="D")
)
display(pd.DataFrame({
    "observed": time_series,
    "linear_interpolation": time_series.interpolate(method="time")
}))

## 13. Group-based imputation
**Use when:** missing values can reasonably be estimated within a meaningful group—for example, median salary within a job level or median temperature per sensor.

**Avoid when:** groups are too small, groups are poorly defined, the group variable leaks the target, or within-group differences are still large.

**Advantages:** respects group differences better than one global statistic.

**Limitations:** small groups can produce unstable estimates; unseen groups require a fallback. Calculate group statistics from training data only.

In [ ]:
group_df = pd.DataFrame({
    "department": ["Sales", "Sales", "IT", "IT", "HR", "HR"],
    "salary": [40000, np.nan, 65000, 70000, np.nan, 38000]
})
group_df["salary_group_median"] = group_df["salary"].fillna(
    group_df.groupby("department")["salary"].transform("median")
)
display(group_df)

## 14. KNN imputation
K-Nearest Neighbors (KNN) imputation estimates a missing value from similar rows.

**Use when:** enough relevant numeric features exist and similar observations are meaningful.

**Avoid when:** the dataset is very large, features are not scaled appropriately, missingness is extensive, or distance between rows has little meaning.

**Advantages:** can capture local patterns and interactions beyond a single global statistic.

**Limitations:** can be computationally expensive, sensitive to feature scaling and irrelevant features, and difficult with mixed data types. Fit within the training pipeline.

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

knn_data = pd.DataFrame({
    "age": [25, 30, np.nan, 40, 35],
    "income": [30000, 40000, 42000, np.nan, 50000]
})
# Scale before distance-based imputation when features have different units.
scaler = StandardScaler()
scaled = scaler.fit_transform(knn_data)
knn = KNNImputer(n_neighbors=2)
imputed_scaled = knn.fit_transform(scaled)
imputed = scaler.inverse_transform(imputed_scaled)
display(pd.DataFrame(imputed, columns=knn_data.columns))

## 15. Iterative imputation
Iterative imputation models each feature with missing values using the other features, repeating the process over multiple rounds.

**Use when:** relationships among numeric variables are informative and a more model-based approach is justified.

**Avoid when:** data are too small or unstable, assumptions are unsuitable, or a simple robust baseline is sufficient. Be careful with categorical features and nonlinear relationships.

**Advantages:** can preserve multivariable relationships better than univariate imputation.

**Limitations:** slower, more complex, model-dependent, and not guaranteed to be unbiased. Evaluate stability and use training-only fitting.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

iter_data = pd.DataFrame({
    "age": [25, 30, np.nan, 40, 35, 50],
    "income": [30000, 40000, 42000, np.nan, 50000, 70000],
    "score": [60, 65, 70, 80, np.nan, 90]
})
iter_imputer = IterativeImputer(random_state=42, max_iter=10)
iter_result = iter_imputer.fit_transform(iter_data)
display(pd.DataFrame(iter_result, columns=iter_data.columns))

## 16. Add missingness indicators
Sometimes the fact that a value was missing is predictive. An indicator can preserve that information, but it should be justified and evaluated on held-out data.

In [ ]:
example = pd.DataFrame({"income": [30000, np.nan, 45000, np.nan, 60000]})
example["income_was_missing"] = example["income"].isna().astype("int8")
example["income_filled"] = example["income"].fillna(example["income"].median())
display(example)

## 17. Compare methods thoughtfully
There is no single best imputation technique. Select a method based on:
- Feature type and distribution
- Missingness amount and likely mechanism
- Time/order and entity structure
- Whether missingness is informative
- Dataset size and computational cost
- Downstream task and validation performance
- Whether the approach can be applied consistently to future data

For a fair comparison, evaluate alternative preprocessing pipelines using training data and cross-validation. Keep the test set untouched until final evaluation.

## 18. A safe preprocessing pipeline example

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

data = pd.DataFrame({
    "age": [22, 35, np.nan, 48, 29, 41, 37, np.nan, 52, 26, 33, 45],
    "city": ["Chennai", "Salem", "Chennai", None, "Madurai", "Salem",
             "Chennai", "Madurai", "Salem", "Chennai", None, "Madurai"],
    "income": [25000, 52000, 33000, 71000, np.nan, 65000,
               59000, 42000, 82000, 28000, 47000, 68000],
    "target": [0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1]
})
X = data[["age", "city", "income"]]
y = data["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
prep = ColumnTransformer([
    ("numeric", numeric_pipe, ["age", "income"]),
    ("categorical", categorical_pipe, ["city"])
])
pipeline = Pipeline([
    ("preprocess", prep),
    ("model", LogisticRegression(max_iter=1000))
])
pipeline.fit(X_train, y_train)
print("Held-out accuracy:", pipeline.score(X_test, y_test))